# FINA4030A — Lab 4
## Two ways to measure tone, and the filings where they disagree

**Class 4.** Submit this notebook by 23:59 on **8 October**.

> **Before you type anything: File → Save a copy in Drive.**

Module 2 starts here. From now on you are building measures, not just testing
somebody else's.

Today you build the same measure twice — once with a word list from 2011 that
costs nothing, once with a language model that costs real money — over the same
corpus of MD&A sections.

**The correlation between them is not the deliverable.** If the two agree, you
have learned nothing you could not have had from the free one. **The deliverable
is every filing where they disagree**, opened and read by you, and classified.

**What is marked.** The disagreement table and what you concluded from it. A
notebook reporting r = 0.87 and stopping scores badly. A notebook that reads six
filings carefully and explains four of them scores well.

In [ ]:
# Setup. Run this first.

REQUIRED_CLIENT = "1.1"
REPO = "https://raw.githubusercontent.com/fy-ericlam/fina4030a/main"

import importlib, sys, urllib.request

urllib.request.urlretrieve(f"{REPO}/fina4030a.py", "fina4030a.py")
urllib.request.urlretrieve(f"{REPO}/labs/lab04_corpus.json", "lab04_corpus.json")

sys.modules.pop("fina4030a", None)
import fina4030a
importlib.reload(fina4030a)

if fina4030a.__version__ < REQUIRED_CLIENT:
    print(f"!! Loaded client v{fina4030a.__version__}, needs v{REQUIRED_CLIENT}.")
    print("   Runtime > Restart session, then run this cell again.")
else:
    print(f"client v{fina4030a.__version__} loaded")

# --- your details -----------------------------------------------------------
NAME       = ""
STUDENT_ID = ""

fina4030a.configure(provider="cuhk_portal")
fina4030a.verify()

---

## The instrument you have to beat

The Loughran–McDonald word lists are the standard baseline for tone in financial
disclosure, and they are not on our repository — you fetch them from the authors,
at Notre Dame, every time you run this notebook.

That is deliberate on two counts. The lists are free for academic use but not
ours to redistribute; and **you should see where your instrument comes from.** A
measurement whose provenance you cannot state is not a measurement you can
defend.

The cell below prints the citation. Read it. It goes in your write-up.

In [ ]:
# The Loughran-McDonald Master Dictionary.
#
# Paste the download link from
#     https://sraf.nd.edu/loughranmcdonald-master-dictionary/
# exactly as it appears --- including a Google Drive "sharing" link, which is
# what the authors currently use. The loader rewrites it to a direct download.
# LM_MIRROR is the Blackboard copy, used if the first fails.

LM_URL    = ""      # the link from sraf.nd.edu, as-is
LM_MIRROR = ""      # a fallback: another shared link, OR the name of a file you
                    # uploaded to the Files pane on the left, e.g.
                    # "Loughran-McDonald_MasterDictionary_1993-2025.csv"

CITATION = ("Loughran, T. and B. McDonald (2011), 'When Is a Liability Not a "
            "Liability? Textual Analysis, Dictionaries, and 10-Ks', Journal of "
            "Finance 66(1), 35-65. Master Dictionary retrieved from "
            "https://sraf.nd.edu/loughranmcdonald-master-dictionary/")

import csv, io, re, urllib.request, zipfile

_GDRIVE = re.compile(r"/file/d/([A-Za-z0-9_-]{10,})|[?&]id=([A-Za-z0-9_-]{10,})")

def direct_url(url):
    """A Drive 'sharing' link serves an HTML preview page, not the file."""
    if "drive.google.com" in url or "drive.usercontent.google.com" in url:
        m = _GDRIVE.search(url)
        if m:
            fid = m.group(1) or m.group(2)
            return ("https://drive.usercontent.google.com/download"
                    f"?id={fid}&export=download&confirm=t")
    return url

def fetch_bytes(src):
    """Accepts a URL or a local path, so a file dragged into the Files pane
    works as a fallback without anyone having to write a file:// URL."""
    if not re.match(r"^[a-z][a-z0-9+.-]*://", src, re.I):
        with open(src, "rb") as fh:                  # a path, not a URL
            raw = fh.read()
    elif src.lower().startswith("file://"):
        with open(src[7:], "rb") as fh:
            raw = fh.read()
    else:
        req = urllib.request.Request(direct_url(src),
                                     headers={"User-Agent": "FINA4030A lab"})
        raw = urllib.request.urlopen(req, timeout=90).read()
    if raw[:2] == b"PK":                       # a zip, as the authors sometimes post
        z = zipfile.ZipFile(io.BytesIO(raw))
        names = [n for n in z.namelist() if n.lower().endswith(".csv")]
        if not names:
            raise RuntimeError(f"Zip contains no CSV: {z.namelist()[:5]}")
        raw = z.read(names[0])
    head = raw[:300].lstrip().lower()
    if head.startswith(b"<!doctype") or head.startswith(b"<html"):
        raise RuntimeError(
            f"That link returned a web page ({len(raw):,} bytes of HTML), not a "
            "file. If it is a Google Drive link, make sure the file is shared as "
            "'Anyone with the link'. Otherwise download it yourself and use the "
            "Blackboard copy in LM_MIRROR.")
    return raw

def load_lm(urls):
    """Return ({category: set(words)}, n_rows) from the Master Dictionary CSV.

    A category column holds the YEAR a word entered that list, so any non-zero
    value means the word is on it. Header spellings have changed between
    releases, so match case- and separator-insensitively rather than exactly.
    """
    last = None
    for url in [u for u in urls if u and u.strip()]:
        try:
            raw = fetch_bytes(url)
            break
        except Exception as e:
            last = f"{type(e).__name__}: {e}"
    else:
        raise RuntimeError(
            "Could not load the Master Dictionary.\n"
            f"Last error: {last}\n\n"
            "Do this instead, it takes a minute:\n"
            "  1. Download the dictionary from the Class 4 folder on Blackboard.\n"
            "  2. Drag it into the Files pane on the left of this notebook.\n"
            "  3. Set LM_MIRROR to the file name, then run this cell again.\n\n"
            "A Blackboard link cannot go in LM_MIRROR: it sits behind CUHK "
            "login, so this notebook cannot fetch it.")

    rows = list(csv.DictReader(io.StringIO(raw.decode("utf-8-sig", "replace"))))
    if not rows:
        raise RuntimeError("Downloaded file is empty or not a CSV.")

    def norm(s): return "".join(ch for ch in s.lower() if ch.isalnum())
    cols = {norm(k): k for k in rows[0]}
    wordcol = cols.get("word")
    if wordcol is None:
        raise RuntimeError(f"No 'Word' column. Columns found: {list(rows[0])[:8]}")

    lists = {}
    for w in ("negative", "positive", "uncertainty", "litigious",
              "strongmodal", "weakmodal", "constraining"):
        col = cols.get(w)
        if col is None:
            continue
        s = set()
        for r in rows:
            v = (r.get(col) or "0").strip()
            try:
                on = float(v) != 0
            except ValueError:
                on = False
            if on:
                s.add(r[wordcol].strip().lower())
        lists[w] = s
    return lists, len(rows)

LM, n_rows = load_lm([LM_URL, LM_MIRROR])
print(f"Master Dictionary: {n_rows:,} words")
for k in sorted(LM):
    print(f"   {k:<13} {len(LM[k]):>6,}")
print()
print("Citation for your write-up:")
print(" ", CITATION)

### The baseline measure

The convention in the literature is the *proportion* of negative words, not the
count — long filings would otherwise look angrier than short ones.

We report three numbers: percent negative, percent positive, and a signed net
tone so that the scale is comparable with what you are about to ask the model
for. Nothing here is clever. That is the point: you can read every line of it,
and it will give the same answer forever.

In [ ]:
import json, re
import pandas as pd

corpus = json.load(open("lab04_corpus.json", encoding="utf-8"))
DOCS = corpus["documents"]
print(f"{len(DOCS)} filings, {corpus['words_per_excerpt']} words each\n")

TOKEN = re.compile(r"[A-Za-z][A-Za-z'-]*")

def dict_tone(text, lists=LM):
    w = [t.lower() for t in TOKEN.findall(text)]
    n = len(w) or 1
    neg = sum(1 for t in w if t in lists.get("negative", ()))
    pos = sum(1 for t in w if t in lists.get("positive", ()))
    unc = sum(1 for t in w if t in lists.get("uncertainty", ()))
    return {"words": len(w),
            "pct_neg": 100 * neg / n,
            "pct_pos": 100 * pos / n,
            "pct_unc": 100 * unc / n,
            "net_tone": 100 * (pos - neg) / n}

base = pd.DataFrame([{**{"id": d["id"], "company": d["company"]},
                      **dict_tone(d["text"])} for d in DOCS])
base = base.sort_values("net_tone").reset_index(drop=True)
print(base[["company", "pct_neg", "pct_pos", "net_tone"]].round(2).to_string(index=False))

---

## The same measure, asked of a model

**One call per filing, and that is the constraint that shapes this cell.**

Scoring the whole corpus would cost one call per document and, at the rate limit
the client throttles itself to, most of the lab block. It would also take a large
bite out of your weekly quota for a result you can get from a sample.

So: the dictionary scores **every** filing, because it is free. The model scores
a **random sample**, drawn with a fixed seed so that everyone in the room can
compare like with like.

Notice what you have just done. You reached for the cheap measure at full
coverage and the expensive one at partial coverage, without being told to. Hold
that thought until the end of class.

In [ ]:
import numpy as np

MODEL_SAMPLE = 12          # calls. Raise it only if your quota allows.
rs = np.random.RandomState(4030)     # stream-stable across numpy versions
pick = sorted(rs.choice(len(DOCS), size=min(MODEL_SAMPLE, len(DOCS)), replace=False))
print("sampled:", ", ".join(DOCS[i]["company"] for i in pick))

PROMPT = """You are reading an excerpt from the Management's Discussion and
Analysis section of a US 10-K filing.

Rate the overall tone of the disclosure on a scale from -5 to +5, where -5 is
uniformly negative about the business and +5 is uniformly positive. Judge the
disclosure's tone, not whether the company is a good investment.

Give one sentence of reasoning. Then finish with exactly this line and nothing
after it:

TONE: <a number from -5 to +5>

EXCERPT:
"""

def tag_num(text, name="TONE"):
    m = re.search(rf"^\s*{name}\s*:\s*([+-]?\d+(?:\.\d+)?)", text,
                  re.MULTILINE | re.IGNORECASE)
    return float(m.group(1)) if m else None

rows, unparsed = [], 0
for k, i in enumerate(pick, 1):
    d = DOCS[i]
    try:
        resp = fina4030a.complete(PROMPT + d["text"], temperature=0.0, max_tokens=300)
        score = tag_num(resp)
    except Exception as e:
        print(f"  {k:>2}/{len(pick)} {d['company']:<20} FAILED — {type(e).__name__}")
        rows.append({"id": d["id"], "company": d["company"], "model_tone": None,
                     "raw": ""})
        continue
    if score is None:
        unparsed += 1
    print(f"  {k:>2}/{len(pick)} {d['company']:<20} "
          f"{'TONE ' + format(score, '+.1f') if score is not None else 'no TONE line'}")
    rows.append({"id": d["id"], "company": d["company"], "model_tone": score,
                 "raw": resp})

model = pd.DataFrame(rows)
print(f"\n{len(model)} scored, {unparsed} without a parseable TONE line.")
if unparsed:
    print("Responses you cannot parse are data, not accidents. Report the count.")

---

## Where they disagree

Both measures are on different scales, so rank them and compare positions. A
filing that the dictionary puts near the bottom and the model puts near the top
is where the interesting thing is.

In [ ]:
cmp = model.merge(base, on=["id", "company"]).dropna(subset=["model_tone"])
cmp["rank_dict"]  = cmp["net_tone"].rank()
cmp["rank_model"] = cmp["model_tone"].rank()
cmp["gap"] = (cmp["rank_model"] - cmp["rank_dict"]).abs()
cmp = cmp.sort_values("gap", ascending=False).reset_index(drop=True)

r = cmp["net_tone"].corr(cmp["model_tone"])
print(f"Spearman-style rank correlation over {len(cmp)} filings: "
      f"{cmp['rank_dict'].corr(cmp['rank_model']):.2f}")
print(f"Correlation of the raw scores: {r:.2f}")
print()
print("Most disagreement first:")
print(cmp[["company", "net_tone", "model_tone", "rank_dict", "rank_model", "gap"]]
      .round(2).to_string(index=False))

print("\nThe correlation above is the least interesting number in this notebook.")
print("Open the top three filings and read them.")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.scatter(cmp["net_tone"], cmp["model_tone"], s=42, color="#1F3B73", zorder=3)
for _, row in cmp.head(3).iterrows():
    ax.scatter([row["net_tone"]], [row["model_tone"]], s=110, facecolors="none",
               edgecolors="#B03A2E", linewidths=1.6, zorder=4)
    ax.annotate(row["company"], (row["net_tone"], row["model_tone"]),
                textcoords="offset points", xytext=(7, 4), fontsize=8,
                color="#B03A2E")
ax.set_xlabel("dictionary net tone  (% positive - % negative)")
ax.set_ylabel("model tone  (-5 to +5)")
ax.set_title("The three circled filings are your assignment", fontsize=10,
             color="#1F3B73")
ax.grid(alpha=.25); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

---

## Read them

Open the text of the three filings with the largest rank gap and read the
excerpt. Not the score — the words.

For each, decide which measure is closer to what the disclosure actually says,
and why. Then classify it:

| Classification | Means |
|---|---|
| `dictionary wrong` | The word counts were misled — accounting vocabulary, negation, boilerplate |
| `model wrong` | The model smoothed over something the words plainly say |
| `both defensible` | Two readings, both supportable. Say what the ambiguity is |
| `genuinely ambiguous` | The disclosure does not have a well-defined tone |

That last row is not a cop-out. If a measure is undefined for some of its
inputs, and does not tell you which ones, that is the single most important fact
about it.

In [ ]:
# Print the excerpts you need to read.
for _, row in cmp.head(3).iterrows():
    d = next(x for x in DOCS if x["id"] == row["id"])
    print("=" * 78)
    print(f"{d['company']}   filed {d['filed']}")
    print(f"dictionary net tone {row['net_tone']:+.2f}   model tone "
          f"{row['model_tone']:+.1f}   rank gap {row['gap']:.0f}")
    print(f"source: {d['source_url']}")
    print("-" * 78)
    print(d["text"][:2200], "...")
    print()

In [ ]:
DISAGREEMENTS = [
    # one dict per filing you read. Copy this block twice more.
    {
        "company":        "",
        "classification": "",   # dictionary wrong | model wrong | both defensible | genuinely ambiguous
        "evidence":       "",   # quote the words that decided it for you
        "why":            "",   # one or two sentences
    },
]

_ok = [d for d in DISAGREEMENTS
       if all(str(d.get(k, "")).strip() for k in
              ("company", "classification", "evidence", "why"))]
VALID = {"dictionary wrong", "model wrong", "both defensible", "genuinely ambiguous"}
_bad = [d["classification"] for d in _ok if d["classification"].strip().lower() not in VALID]

if len(_ok) < 3:
    print(f"{len(_ok)} of 3 filings written up.")
elif _bad:
    print("Classification must be one of:", ", ".join(sorted(VALID)))
    print("You wrote:", _bad)
else:
    from collections import Counter
    tally = Counter(d["classification"].strip().lower() for d in _ok)
    print("Your tally:")
    for k, v in tally.most_common():
        print(f"   {k:<22} {v}")
    if tally.get("genuinely ambiguous"):
        print("\nYou found a filing where the quantity itself is not well defined.")
        print("Say in your findings what a firm should do with a measure that is")
        print("undefined for some inputs and does not flag which ones.")

---

## What it would cost at scale

Roughly seven thousand 10-K filings are lodged with the SEC each year. You have
just scored twelve excerpts.

In [ ]:
calls = [c for c in fina4030a.TRANSCRIPT if not c.error and c.usage]
tok = sum(c.usage.get("total_tokens", 0) for c in calls)
sec = sum(c.seconds for c in calls)

if calls:
    per_tok, per_sec = tok / len(calls), sec / len(calls)
    print(f"{len(calls)} scoring calls: {tok:,} tokens, {sec:.0f}s")
    print(f"per filing      : {per_tok:,.0f} tokens, {per_sec:.1f}s")
    print()
    for n, label in ((7000, "one year of US 10-Ks"),
                     (70000, "ten years")):
        print(f"{label:<22} {n * per_tok / 1e6:>8,.1f}M tokens   "
              f"{n * per_sec / 3600:>7,.1f} hours of wall clock")
    print()
    print("The dictionary would do the same corpus in under a minute, on a laptop,")
    print("for nothing. Write down at what corpus size you would stop using the")
    print("model, and what you would use it for instead of abandoning it.")
else:
    print("No successful scoring calls recorded.")

---

## Findings

In [ ]:
FINDINGS = {
    "what_the_correlation_was": "",  # one line, and why it is not the answer
    "most_interesting_filing":  "",  # which, and what you concluded
    "which_measure_you_trust":  "",  # for what purpose. "It depends" needs a "on what".
    "the_architecture":         "",  # cheap measure everywhere, expensive where?
                                     # Say where the boundary is and how you would set it.
    "unparseable_responses":    None,  # how many, as an integer
    "confidence":               None,  # 1-5
}

_missing = [k for k, v in FINDINGS.items()
            if v is None or (isinstance(v, str) and not v.strip())]
print("Complete." if not _missing else "Still to fill in: " + ", ".join(_missing))

In [ ]:
print(fina4030a.appendix(
    student=f"{NAME} ({STUDENT_ID})",
    verification=FINDINGS.get("most_interesting_filing", ""),
    residual_risk=FINDINGS.get("which_measure_you_trust", ""),
    reproducibility=(
        f"Dictionary measure over all {len(DOCS)} corpus documents using the "
        f"Loughran-McDonald Master Dictionary. Model measure over a "
        f"RandomState(4030) sample of {MODEL_SAMPLE}. Temperature 0.0 requested. "
        f"Corpus: cached MD&A excerpts, {corpus['words_per_excerpt']} words each."),
))
print()
print("Dictionary citation:", CITATION)

fina4030a.save_transcript("lab04_transcript.json")

---

## What to submit

1. This notebook with outputs intact.
2. `lab04_transcript.json`.
3. The appendix printed above, including the dictionary citation.

## One thing to carry forward

You built two measures. One is transparent, free, reproducible and crude. The
other is subtle, expensive, unreproducible and cannot explain itself.

Neither is *the* answer, and the useful skill is not picking a side — it is
knowing which properties you are buying and which you are giving up, and being
able to say so to whoever signs off on the number.

**And one new idea from today: the prompt is part of the instrument.** Change the
wording and you have changed the measurement. If you do not version your prompts,
you cannot reproduce your own results — and this will be on the Class 8
assessment.

---

### Next

**Class 5** is valuation and financial modelling. The arithmetic from Class 2
comes back, with money attached.